# CRAFT Phase 0: Capability Probe
This notebook runs the capability probe on Phi-3-Mini to build the offline `difficulty_map.json`.
It measures the pass@5 rate on GSM8K, AQuA-RAT, StrategyQA, and MMLU questions (total 200 samples).

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
git_token = user_secrets.get_secret("GITHUB_PAT")
username = "VishalGastu30"

# Clone the repo using the token
!git clone https://{git_token}@github.com/{username}/CRAFT-SLM-Reasoning.git

# Move into the directory
%cd CRAFT-SLM-Reasoning

In [ ]:
# 1. Install dependencies
!pip install -q transformers bitsandbytes accelerate datasets loguru pyyaml scipy numpy

In [ ]:
!pip install -q transformers==4.44.0 trl==0.10.1 peft==0.12.0 bitsandbytes==0.41.3 accelerate==0.34.2 datasets==2.20.0
!pip install -q lm-eval==0.4.3 llama-cpp-python loguru rich triton

In [ ]:
# 2. Verify files and structure
import os
assert os.path.exists("src/phase0_probe/probe.py"), "Please make sure you run this notebook from the root directory of the cloned repository!"

In [ ]:
# Run this cell to patch probe.py and remove the unnecessary bitsandbytes dependency
file_path = 'src/phase0_probe/probe.py'
with open(file_path, 'r') as f:
    content = f.read()

# Replace the 4-bit quantization flag with standard bfloat16
content = content.replace(
    'torch_dtype=torch.float16,\n                    load_in_4bit=True', 
    'torch_dtype=torch.bfloat16,\n                    trust_remote_code=True'
)

with open(file_path, 'w') as f:
    f.write(content)
print("probe.py patched! You can now run Phase 0.")

In [ ]:
# 3. Run capability probe
!python -m src.phase0_probe.probe --config phi3_mini --hardware kaggle --output difficulty_map.json

In [ ]:
# 4. Display results summary
import json
if os.path.exists("difficulty_map.json"):
    with open("difficulty_map.json", "r") as f:
        dmap = json.load(f)
    print(f"Total probed questions: {len(dmap)}")
    diffs = [v['difficulty'] for v in dmap.values()]
    print(f"Easy (<=0.3): {sum(1 for d in diffs if d <= 0.3)}")
    print(f"Medium (0.4-0.7): {sum(1 for d in diffs if 0.4 <= d <= 0.7)}")
    print(f"Hard (>=0.8): {sum(1 for d in diffs if d >= 0.8)}")
else:
    print("Error: difficulty_map.json not found!")